<a href="https://colab.research.google.com/github/Bassendiaye/mes_notebooks/blob/main/Bassendiaye/mes_notebooks/VGG19_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import random
import numpy as np
from uuid import uuid4
from tqdm import tqdm
from collections import defaultdict
from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models

In [ ]:
def prepare_data_with_oversampling(data_dir, test_size=0.3, random_state=42, max_samples_per_class=4000):

    IMG_HEIGHT = 46
    IMG_WIDTH  = 322

    class_names = sorted([d for d in os.listdir(data_dir)
                          if os.path.isdir(os.path.join(data_dir, d))])

    file_paths, labels = [], []
    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)
        class_files = [
            os.path.join(class_dir, f)
            for f in os.listdir(class_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        file_paths.extend(class_files)
        labels.extend([class_idx] * len(class_files))

    train_files, val_files, train_labels, val_labels = train_test_split(
        file_paths, labels, test_size=test_size,
        stratify=labels, random_state=random_state
    )

    def get_individual_transforms():
        return [
            A.HorizontalFlip(p=1),
            A.VerticalFlip(p=1),
            A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=1),
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=1),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
            A.RGBShift(r_shift_limit=20, g_shift_limit=20, b_shift_limit=20, p=1),
            A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=20, val_shift_limit=15, p=1),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1),
            A.Defocus(radius=(3, 5), p=1),
            A.MotionBlur(blur_limit=(3, 5), p=1),
            A.GaussianBlur(blur_limit=(3, 5), p=1),
        ]

    def oversample(files, labels):
        label_to_files = defaultdict(list)
        for f, l in zip(files, labels):
            label_to_files[l].append(f)

        transforms = get_individual_transforms()

        base_transform = A.Compose([
            A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
            A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
            ToTensorV2()
        ])

        AUG_DIR = os.path.join(os.path.dirname(data_dir), "split_4000_images_balanced")
        os.makedirs(AUG_DIR, exist_ok=True)

        oversampled_files, oversampled_labels = [], []

        for label, file_list in tqdm(label_to_files.items(), desc="Balancing classes"):
            n = len(file_list)

            if n >= max_samples_per_class:
                sampled = random.sample(file_list, max_samples_per_class)
                oversampled_files.extend(sampled)
                oversampled_labels.extend([label] * max_samples_per_class)

            else:
                oversampled_files.extend(file_list)
                oversampled_labels.extend([label] * n)

                needed = max_samples_per_class - n
                idx = 0

                while needed > 0:
                    img_path = file_list[idx % n]
                    image = cv2.imread(img_path)
                    if image is None:
                        idx += 1
                        continue
                    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                    for t in transforms:
                        if needed <= 0: break

                        composed = A.Compose([t, base_transform])
                        augmented = composed(image=image)
                        aug = augmented["image"]

                        aug_np = aug.permute(1, 2, 0).cpu().numpy()
                        aug_np = np.clip((aug_np * 0.5 + 0.5) * 255, 0, 255).astype("uint8")
                        aug_bgr = cv2.cvtColor(aug_np, cv2.COLOR_RGB2BGR)

                        new_path = os.path.join(AUG_DIR, f"{uuid4().hex}.jpg")
                        cv2.imwrite(new_path, aug_bgr)

                        oversampled_files.append(new_path)
                        oversampled_labels.append(label)
                        needed -= 1

                    idx += 1

        return oversampled_files, oversampled_labels

    train_files, train_labels = oversample(train_files, train_labels)

    return train_files, val_files, train_labels, val_labels, class_names

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, image_paths, labels, is_train=True):
        self.paths = image_paths
        self.labels = labels
        self.is_train = is_train

        self.train_transform = A.Compose([
            A.Resize(224, 224),
            A.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
            ToTensorV2()
        ])

        self.val_transform = A.Compose([
            A.Resize(224, 224),
            A.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        label    = self.labels[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.is_train:
            image = self.train_transform(image=image)["image"]
        else:
            image = self.val_transform(image=image)["image"]

        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
data_dir = "/content/drive/MyDrive/Dossier_de_Basse/Data_paper_T_V_T/train_Val"

train_files, val_files, train_labels, val_labels, class_names = \
    prepare_data_with_oversampling(data_dir)

train_dataset = CustomImageDataset(train_files, train_labels, is_train=True)
val_dataset   = CustomImageDataset(val_files, val_labels, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Classes:", class_names)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))


In [ ]:
num_classes = len(class_names)

vgg19 = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
vgg19.classifier[6] = nn.Linear(4096, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vgg19 = vgg19.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg19.parameters(), lr=1e-4)

best_val_acc = 0

for epoch in range(25):

    # ------------------ TRAIN -------------------
    vgg19.train()
    correct = total = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = vgg19(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = correct / total * 100

    # ------------------ VALIDATION -------------------
    vgg19.eval()
    correct = total = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = vgg19(imgs)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total * 100

    print(f"Epoch {epoch+1}/25 - Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    # ---------- Sauvegarde meilleur modèle ----------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(vgg19.state_dict(), "Dossier_de_Basse/split_best_vgg19.pth")
        print(f" Meilleur modèle sauvegardé ! Avec best_val_acc = {best_val_acc: .2f}% ")

In [ ]:
vgg.load_state_dict(torch.load(best_model_path))
vgg.eval()

test_preds, test_true = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = vgg(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        test_preds.extend(preds)
        test_true.extend(labels.numpy())

acc = accuracy_score(test_true, test_preds)
cm = confusion_matrix(test_true, test_preds)

print("Test Accuracy =", acc)
print("Confusion Matrix:\n", cm)